# FANET Simulator Guide: MAGAT-D3QN & Single-Agent Baselines

Welcome to the official run guide for the **Mobility and Channel-Aware Multi-Agent FANET Simulator**. 

This notebook explains the core files, configuration steps, and terminal execution commands to properly train your Reinforcement Learning algorithms and evaluate their performance over dynamic topologies.

## 1. Environment Setup
Ensure you have installed all necessary dependencies before executing the server suite. This project requires `PyTorch Geometric`, `PettingZoo`, and `Stable-Baselines3`.

```bash
pip install -r requirements.txt
```

## 2. Configuration (`configs/config.py`)

Before running any script, set your desired physics bounds in `configs/config.py`. 

- **`N`**: Number of UAVs in the swarm (e.g., 150).
- **`MOBILITY_MODEL`**: The flight path behavior (e.g., `"circular"`, `"gauss_markov"`).
- **`SWEEP_MAX_PPS`**: The highest traffic load threshold you want to evaluate (e.g., `1000`).
- **`RUN_MARL_GNN`**: Set this to `True` if you want to include the PyTorch MAGAT-D3QN model in the final evaluation graphs.

## 3. Training the Multi-Agent Attention Model (MAGAT-D3QN)

Since our proposed model uses a custom PyTorch Multi-Head Attention layer (`GATConv`) combined with Dueling value streams, it must be trained natively outside of the standard loop. 

1. Open `training/train_marl.py`
2. Set `episodes = 5000` (or `10000` for final IEEE weights).
3. Run the following cell, or execute this command in your terminal:

In [ ]:
!python training/train_marl.py

*(This produces `results/checkpoints/gnn_marl_model.pth`)*

## 4. Full Server Suite Evaluation (Baselines + MARL)

Once the `.pth` weights are saved, you can run the primary orchestration loop. This script (`experiments/run_experiments_server.py`) will:
1. Span an isolated Multiprocessing pool utilizing all of your CPU cores.
2. Automatically train the single-agent baselines (`Tabular`, `DQN`, `PPO`, `A2C`, `MCA-D3QN`) over `50000` timesteps.
3. Load your pre-trained `gnn_marl_model.pth`.
4. Iteratively evaluate every algorithm against identical traffic loads to guarantee perfect 1:1 fairness.
5. Plot the aggregated Line Graphs (Throughput and Drop Rates) comparing all the models directly!

In [ ]:
!python experiments/run_experiments_server.py

## 5. Locating Your Results
All output artifacts are saved cleanly inside the `results/` root directory dynamically mapped by timestamps.

- **Graphs**: `results/{CONFIG_NAME}/trial_{TIMESTAMP}/RL_comparison/images/aggregate_throughput_bars.png`
- **CSVs**: `results/{CONFIG_NAME}/trial_{TIMESTAMP}/RL_comparison/csv/model_performance_benchmarks.json`
- **Weights**: `results/checkpoints/` (These overwrite natively, so do not push them to Git).